# P08B — Discovery dan inferensi kausal

Jalankan P08A dahulu. ATE = E[Y(1)-Y(0)] untuk IRI enam bulan. Nilai negatif berarti kondisi lebih baik.

In [ ]:
# Folder kerja: jalankan dari folder mana pun. Sel ini pindah ke akar repositori agar jalur data relatif (praktikum/...) benar.
import os
from pathlib import Path
_akar = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "praktikum").is_dir() and (p / "notebooks").is_dir()), None)
if _akar is not None:
    os.chdir(_akar)
else:
    print("Akar repositori tidak ditemukan; memakai folder kerja saat ini (normal di Colab).")

In [ ]:
# Data untuk Colab: bila berkas data tidak ada (mis. notebook dibuka di tab atau runtime baru), unduh dari repositori.
# Pada komputer dengan repositori hasil clone, atau bila hasil notebook pembangkit (A) sudah ada, sel ini tidak mengunduh apa pun.
import urllib.request
from pathlib import Path
_BASE = "https://raw.githubusercontent.com/Ryandran/Repository_ML_Rekayasa_Sipil/main/"
_PERLU = [  # (jalur di repositori, jalur lain yang juga diterima)
    ('praktikum/bab08/maintenance_synthetic.csv', ['maintenance_synthetic.csv']),
]
for _jalur, _alt in _PERLU:
    if any(Path(p).exists() for p in [_jalur, *_alt]):
        continue
    Path(_jalur).parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(_BASE + _jalur, _jalur)
        print("Diunduh dari repositori:", _jalur)
    except Exception as _e:
        print("Gagal mengunduh", _jalur, "->", _e, "| jalankan notebook pembangkit data (A) pada runtime yang sama atau unggah berkas itu.")


In [1]:
from pathlib import Path
import pandas as pd
f=Path('praktikum/bab08/maintenance_synthetic.csv'); print(pd.read_csv(f).describe().round(2))

       iri_baseline  truck_share  annual_rainfall_mm  drainage_risk  \
count       2400.00      2400.00             2400.00        2400.00   
mean           4.05         0.27             1796.22           0.32   
std            0.99         0.14              426.12           0.47   
min            1.20         0.03              700.00           0.00   
25%            3.40         0.16             1505.50           0.00   
50%            4.05         0.25             1791.00           0.00   
75%            4.72         0.36             2085.87           1.00   
max            7.20         0.72             3000.00           1.00   

       maintenance  iri_6_months  
count      2400.00       2400.00  
mean          0.41          4.46  
std           0.49          1.50  
min           0.00          0.50  
25%           0.00          3.48  
50%           0.00          4.50  
75%           1.00          5.46  
max           1.00          9.22  


## Estimasi intervensi: pemeliharaan ruas jalan

Confounder terukur: IRI awal, proporsi truk, curah hujan, risiko drainase. DGP memasukkan efek treatment −0,80 IRI. Perbedaan mentah bukan efek kausal.

In [2]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.linear_model import LinearRegression
f=Path('praktikum/bab08/maintenance_synthetic.csv')
if not f.exists(): f=Path('maintenance_synthetic.csv')
if not f.exists():
 try:
  from google.colab import files
  u=files.upload(); f=Path(next(iter(u)))
 except Exception as e: raise FileNotFoundError('Jalankan P08A atau unggah CSV') from e
df=pd.read_csv(f); C=['iri_baseline','truck_share','annual_rainfall_mm','drainage_risk']; assert df.notna().all().all()
crude=df[df.maintenance==1].iri_6_months.mean()-df[df.maintenance==0].iri_6_months.mean()
X=df[C+['maintenance']]; m=LinearRegression().fit(X,df.iri_6_months); x1=X.copy(); x0=X.copy(); x1.maintenance=1; x0.maintenance=0
ate=np.mean(m.predict(x1)-m.predict(x0)); print('N=',len(df),'raw difference=',round(crude,3),'standardized estimate=',round(ate,3),'DGP truth=-0.80')
rng=np.random.default_rng(20261116); est=[]
for _ in range(500):
 s=df.iloc[rng.integers(0,len(df),len(df))]; xx=s[C+['maintenance']]; mm=LinearRegression().fit(xx,s.iri_6_months); a=xx.copy(); b=xx.copy(); a.maintenance=1; b.maintenance=0; est.append(np.mean(mm.predict(a)-mm.predict(b)))
print('Bootstrap 95% percentile CI:',np.quantile(est,[.025,.975]).round(3))
print('Bootstrap quantifies sampling variation only; it does not remove unmeasured confounding.')

N= 2400 raw difference= 0.453 standardized estimate= -0.828 DGP truth=-0.80


Bootstrap 95% percentile CI: [-0.865 -0.783]
Bootstrap quantifies sampling variation only; it does not remove unmeasured confounding.


## Discovery sebagai ilustrasi terbatas

Skeleton PC berikut menggunakan uji Fisher-Z. Kode tidak menyelesaikan orientasi dan tidak dimaksudkan sebagai implementasi penelitian. Asumsi discovery meliputi kausal sufficiency, faithfulness, tidak ada siklus, sampel memadai, dan uji independensi yang sesuai. Arah collider diketahui dari DGP dan mekanika.

In [3]:
# Demonstrasi skeleton PC sangat terbatas pada tiga variabel kontinu.
# Uji Fisher-Z dan pencarian subset conditioning; tidak mengorientasikan edge, bukan implementasi produksi.
import numpy as np,pandas as pd
from itertools import combinations
from scipy.stats import norm
r=np.random.default_rng(8); n=1500; Z=r.lognormal(0,.35,n); fy=r.lognormal(0,.18,n); M=Z*fy+r.normal(0,.015,n); beam=pd.DataFrame({'Z':Z,'fy':fy,'M':M})
def pcorr(a,b,cond):
 x=beam[a].values; y=beam[b].values
 if cond:
  c=np.column_stack([np.ones(n),beam[list(cond)].values]); x=x-c@np.linalg.lstsq(c,x,rcond=None)[0]; y=y-c@np.linalg.lstsq(c,y,rcond=None)[0]
 rho=np.corrcoef(x,y)[0,1]; z=abs(np.arctanh(np.clip(rho,-.99999,.99999)))*np.sqrt(n-len(cond)-3); return rho,2*(1-norm.cdf(z))
edges={frozenset(e) for e in combinations(beam.columns,2)}
for x,y in combinations(beam.columns,2):
 for cond in [()]+[(q,) for q in beam.columns if q not in (x,y)]:
  if pcorr(x,y,cond)[1]>.05: edges.discard(frozenset((x,y))); break
print('Skeleton:',[sorted(e) for e in edges]); print('No arrow orientation; collider Z -> M <- fy comes from known DGP/domain, not this skeleton.')

Skeleton: [['M', 'fy'], ['M', 'Z']]
No arrow orientation; collider Z -> M <- fy comes from known DGP/domain, not this skeleton.


## Refleksi
- Apakah treatment didefinisikan sebagai jenis/intensitas/waktu pekerjaan tertentu?
- Confounder apa yang mungkin tidak diukur (mis. kualitas kontraktor)?
- Apakah overlap cukup dan apakah ada spillover antarruas?
- Bagaimana nonlinearitas atau confounder hipotetis memengaruhi kesimpulan?
- Bootstrap mengukur variasi sampling, bukan bias identifikasi. DoWhy opsional untuk alur model–identify–estimate–refute; API dan versi perlu dicatat.